In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'pyloudnorm>=0.1.1',
    'faster-whisper>=1.0.0',
    'demucs>=4.0.1',
    'pyannote.audio>=3.3.0',
    'torch>=2.3.0',
    'torchaudio>=2.3.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)

In [ ]:
import os
import re
import json
import time
import threading
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import yaml
import requests
import numpy as np
import soundfile as sf
import pyloudnorm as pyln
import torch
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
SEGMENTS_DIR    = WORK_DIR / 'segments'
SNR_FLAG_DIR    = WORK_DIR / 'snr_flagged'
CLEAN_DIR       = WORK_DIR / 'clean_final'
METADATA_PATH   = WORK_DIR / 'metadata.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1d.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR    = 24000
TARGET_LUFS  = -23.0
SNR_PASS     = 20.0
URDU_CHARS   = set('ابپتثجچحخدذرزژسشصضطظعغفقکگلمنوہھیئاآءۃے')
SAVE_EVERY   = 50
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'[config] device={DEVICE}')

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'GEMINI_API_KEY':  c.get_secret('GEMINI_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')
    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY', 'GEMINI_API_KEY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS     = load_secrets()
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0 repo: {STAGE0_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — done={len(state["done"])} metadata={state["stats"]["metadata_written"]}')
            return state
        except Exception:
            pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1d.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback — done={len(state["done"])}')
            return state
    except Exception:
        pass
    print('[checkpoint] fresh start')
    return {
        'done': [],
        'stats': {
            'demucs_applied': 0,
            'demucs_recovered': 0,
            'lang_reject': 0,
            'whisper_conf_reject': 0,
            'metadata_written': 0,
        },
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload:
        return
    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p1d.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1d checkpoint',
            )
            return
        except Exception:
            time.sleep(min(2 ** attempt, 60))


state    = load_checkpoint()
done_set = set(state['done'])

In [ ]:
whisper_cache = {}

def get_whisper(size='large-v3', device='cuda'):
    key = (size, device)
    if key not in whisper_cache:
        from faster_whisper import WhisperModel
        compute = 'float16' if device == 'cuda' else 'int8'
        whisper_cache[key] = WhisperModel(size, device=device, compute_type=compute)
        print(f'[whisper] {size} loaded on {device}')
    return whisper_cache[key]


def compute_snr(audio, frame_length=2048, noise_percentile=10):
    if len(audio) < frame_length:
        return 0.0
    hop = frame_length // 2
    energies = np.array([
        np.mean(audio[i:i + frame_length] ** 2)
        for i in range(0, len(audio) - frame_length, hop)
    ])
    energies = energies[energies > 0]
    if len(energies) == 0:
        return 0.0
    noise_floor = np.percentile(energies, noise_percentile)
    if noise_floor <= 0:
        return 60.0
    return float(10 * np.log10(np.mean(energies) / noise_floor))


def apply_demucs(input_path, output_path):
    out_dir = Path(output_path).parent / 'demucs_tmp'
    out_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        [
            'python', '-m', 'demucs.separate',
            '--two-stems', 'vocals',
            '--name', 'htdemucs',
            '--device', DEVICE,
            '--out', str(out_dir),
            str(input_path),
        ],
        capture_output=True, timeout=300,
    )
    if result.returncode != 0:
        return False
    vocals = out_dir / 'htdemucs' / Path(input_path).stem / 'vocals.wav'
    if not vocals.exists():
        return False
    subprocess.run(
        ['ffmpeg', '-y', '-i', str(vocals), '-ar', str(TARGET_SR), '-ac', '1', str(output_path)],
        capture_output=True, timeout=60,
    )
    return Path(output_path).exists()


def transcribe(audio_path, device='cuda'):
    model = get_whisper('large-v3', device)
    raw_segs, info = model.transcribe(
        str(audio_path),
        language='ur',
        task='transcribe',
        beam_size=5,
        word_timestamps=True,
        vad_filter=False,
    )
    segs_list  = list(raw_segs)
    full_text  = ' '.join(s.text.strip() for s in segs_list).strip()
    word_ts    = []
    all_probs  = []
    for seg in segs_list:
        if seg.words:
            for w in seg.words:
                word_ts.append({'word': w.word.strip(), 'start': round(w.start, 3),
                                'end': round(w.end, 3), 'probability': round(w.probability, 3)})
                all_probs.append(w.probability)
    avg_conf      = round(float(np.mean(all_probs)), 3) if all_probs else 0.0
    english_words = list(set(w.lower() for w in re.findall(r'\b[a-zA-Z]{2,}\b', full_text)))
    has_urdu      = any(c in URDU_CHARS for c in full_text)
    if not has_urdu and not english_words:
        return None
    return {
        'transcript': full_text,
        'language': info.language,
        'language_probability': round(info.language_probability, 3),
        'avg_confidence': avg_conf,
        'word_timestamps': word_ts,
        'contains_code_switch': len(english_words) > 0,
        'english_words': english_words,
    }


def get_diarization_pipeline():
    if not hasattr(get_diarization_pipeline, '_pipeline'):
        from pyannote.audio import Pipeline
        get_diarization_pipeline._pipeline = Pipeline.from_pretrained(
            'pyannote/speaker-diarization-3.1',
            use_auth_token=HF_TOKEN,
        )
        get_diarization_pipeline._pipeline.to(torch.device(DEVICE))
        print('[diarization] pyannote pipeline loaded')
    return get_diarization_pipeline._pipeline


def get_speaker_id(audio_path, video_id):
    try:
        pipeline   = get_diarization_pipeline()
        diarization = pipeline(str(audio_path))
        speakers   = list({spk for _, _, spk in diarization.itertracks(yield_label=True)})
        if speakers:
            return f'{video_id}_{speakers[0]}'
    except Exception:
        pass
    return f'{video_id}_spk0'


print('[models] loading Whisper large-v3...')
get_whisper('large-v3', DEVICE)

In [ ]:
seg_records_path = WORK_DIR / 'seg_records_pre_lang.jsonl'
seg_records      = []

if seg_records_path.exists():
    with open(seg_records_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                seg_records.append(json.loads(line))

flagged_files = sorted(SNR_FLAG_DIR.glob('*.wav'))
pass_files    = sorted(SEGMENTS_DIR.glob('*.wav'))

all_seg_files = [
    (p, False) for p in pass_files
] + [
    (p, True) for p in flagged_files
]

pending = [(p, is_flagged) for p, is_flagged in all_seg_files if p.stem not in done_set]

print(f'[p1d] pass_segs={len(pass_files)} flagged_segs={len(flagged_files)}')
print(f'[p1d] already done={len(done_set)} pending={len(pending)}')

In [ ]:
seg_lookup = {r['seg_id']: r for r in seg_records}

for idx, (seg_path, is_flagged) in enumerate(pending):
    seg_id   = seg_path.stem
    seg_meta = seg_lookup.get(seg_id, {})
    video_id = seg_meta.get('video_id', seg_id.split('_c')[0])

    try:
        demucs_applied = False
        working_path   = seg_path

        if is_flagged:
            demucs_out = CLEAN_DIR / f'{seg_id}_demucs.wav'
            ok = apply_demucs(seg_path, demucs_out)
            with cp_lock:
                state['stats']['demucs_applied'] += 1

            if ok:
                audio, _ = sf.read(str(demucs_out), dtype='float32')
                snr_after = compute_snr(audio)
                if snr_after >= SNR_PASS:
                    working_path   = demucs_out
                    demucs_applied = True
                    with cp_lock:
                        state['stats']['demucs_recovered'] += 1
                else:
                    demucs_out.unlink(missing_ok=True)
                    with cp_lock:
                        done_set.add(seg_id)
                        state['done'].append(seg_id)
                    continue
            else:
                with cp_lock:
                    done_set.add(seg_id)
                    state['done'].append(seg_id)
                continue

        tx = transcribe(working_path, DEVICE)

        if tx is None:
            with cp_lock:
                state['stats']['lang_reject'] += 1
                done_set.add(seg_id)
                state['done'].append(seg_id)
            continue

        lang = tx['language']
        if lang not in ('ur', 'hi'):
            with cp_lock:
                state['stats']['lang_reject'] += 1
                done_set.add(seg_id)
                state['done'].append(seg_id)
            continue

        if lang == 'hi' and tx['language_probability'] >= 0.85:
            with cp_lock:
                state['stats']['lang_reject'] += 1
                done_set.add(seg_id)
                state['done'].append(seg_id)
            continue

        speaker_id = get_speaker_id(working_path, video_id)

        final_path = CLEAN_DIR / f'{seg_id}.wav'
        if working_path != final_path:
            import shutil
            shutil.copy2(str(working_path), str(final_path))

        audio_for_snr, _ = sf.read(str(final_path), dtype='float32')
        snr_final        = compute_snr(audio_for_snr)
        lang_tag         = 'ur+en' if tx['contains_code_switch'] else lang

        record = {
            'id': seg_id,
            'source': {
                'platform':         'youtube',
                'video_id':         video_id,
                'video_title':      seg_meta.get('video_title', ''),
                'channel_id':       seg_meta.get('channel_id', ''),
                'channel_category': seg_meta.get('channel_category', 'general'),
                'query_used':       seg_meta.get('query_used', ''),
                'download_timestamp': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
            },
            'audio': {
                'path':           f'audio/{seg_id}.wav',
                'duration_sec':   round(seg_meta.get('duration_sec', 0), 3),
                'sample_rate':    TARGET_SR,
                'channels':       1,
                'loudness_lufs':  TARGET_LUFS,
                'snr_db':         round(snr_final, 2),
                'demucs_applied': demucs_applied,
            },
            'speaker': {
                'speaker_id':          speaker_id,
                'gender':              'unknown',
                'estimated_age_group': 'adult',
            },
            'transcript': {
                'urdu_script':           tx['transcript'],
                'language':              lang,
                'language_probability':  tx['language_probability'],
                'whisper_confidence':    tx['avg_confidence'],
                'contains_code_switch':  tx['contains_code_switch'],
                'english_words_detected': tx['english_words'],
                'word_timestamps':        tx['word_timestamps'],
            },
            'domain':  seg_meta.get('channel_category', 'general'),
            'quality': {
                'snr_db':                    round(snr_final, 2),
                'snr_label':                 seg_meta.get('snr_label', 'pass'),
                'snr_passed':                snr_final >= SNR_PASS,
                'lang_id':                   lang_tag,
                'whisper_confidence_passed':  tx['avg_confidence'] >= 0.75,
                'usable_for_codec':          True,
                'usable_for_ce':             tx['avg_confidence'] >= 0.75,
            },
            'split': 'train',
        }

        with open(METADATA_PATH, 'a', encoding='utf-8') as mf:
            mf.write(json.dumps(record, ensure_ascii=False) + '\n')

        with cp_lock:
            state['stats']['metadata_written'] += 1
            done_set.add(seg_id)
            state['done'].append(seg_id)

    except Exception as e:
        print(f'  [error] {seg_id}: {e}')
        with cp_lock:
            done_set.add(seg_id)
            state['done'].append(seg_id)

    if (idx + 1) % SAVE_EVERY == 0 or idx + 1 == len(pending):
        upload_now = (idx + 1) % (SAVE_EVERY * 4) == 0
        save_checkpoint(state, upload=upload_now)
        print(f'  [{idx+1}/{len(pending)}] written={state["stats"]["metadata_written"]} '
              f'lang_reject={state["stats"]["lang_reject"]} '
              f'demucs_recovered={state["stats"]["demucs_recovered"]}')

In [ ]:
total_records = sum(1 for _ in open(METADATA_PATH, encoding='utf-8')) if METADATA_PATH.exists() else 0
total_dur     = 0.0
code_switch   = 0

if METADATA_PATH.exists():
    with open(METADATA_PATH, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                r = json.loads(line)
                total_dur   += r.get('audio', {}).get('duration_sec', 0)
                if r.get('transcript', {}).get('contains_code_switch'):
                    code_switch += 1

print('\n[p1d] final summary')
print(f'  metadata records  : {total_records}')
print(f'  total duration    : {total_dur/3600:.2f}h')
print(f'  code-switched     : {code_switch} ({100*code_switch/max(total_records,1):.1f}%)')
print(f'  demucs applied    : {state["stats"]["demucs_applied"]}')
print(f'  demucs recovered  : {state["stats"]["demucs_recovered"]}')
print(f'  lang rejected     : {state["stats"]["lang_reject"]}')

save_checkpoint(state, upload=True)
print('\n[done] ready for p1e_upload.ipynb')